# Pricing across asset classes

**Prerequisites:** `02_pricing/pricing_fundamentals.ipynb` (the standard pricing pipeline: tagged instrument JSON, `MarketContext` as JSON, valuation date, model key, and `ValuationResult.from_json`). This notebook reuses that same pipeline while swapping the **instrument** and, when needed, the **model** and **metrics**.

**Goal:** see one consistent workflow for rates, credit, equity, and FX—without leaving the JSON-oriented API.

## Concepts: a quick tour of asset classes

Finstack Quant groups instruments by **risk factor** and **cashflow shape**, not by desk org chart. The same pricing entry points apply everywhere:

- **Rates (money market & swaps):** PV from discounting (and forward projection for float legs). Curves: OIS discount, IBOR/RFR **forward** where needed.
- **Credit (CDS):** protection vs premium legs; survival from **hazard** curves. Model key is typically `hazard_rate` rather than pure `discounting`.
- **Equity options:** Black–Scholes–style models (`black76`) with **spot**, **discount**, **dividend yield**, and an **implied vol surface**.
- **FX options:** Garman–Kohlhagen-style stack: **two discount curves** (domestic/foreign), **FX spot** in the matrix, and a **vol surface** keyed by expiry × strike.

The pattern in each section below is deliberate and repeatable:

1. Build tagged instrument JSON (`{"type": "…", "spec": {…}}`).
2. Build a **fresh** typed `MarketContext` and serialize it with `to_json()`.
3. Call `price_instrument` or `price_instrument_with_metrics` with the right **model** string.
4. Parse with `ValuationResult.from_json` and read `price`, `currency`, and optional `get_metric`.

In [1]:
import json
from datetime import date
from pathlib import Path

import sys
sys.path.insert(0, "..")

from _shared import DEMO_AS_OF, build_demo_market
from finstack_quant.core.market_data import PriceCurve, VolSurface
from finstack_quant.valuations import ValuationResult
from finstack_quant.valuations.instruments import (
    list_standard_metrics_grouped,
    price_instrument,
    price_instrument_with_metrics,
    validate_instrument_json,
)

grouped = list_standard_metrics_grouped()
total = sum(len(v) for v in grouped.values())
print(f"Standard metrics ({total}) across {len(grouped)} groups:")
for group, metrics in grouped.items():
    print(f"  {group}: {', '.join(metrics[:5])}{'...' if len(metrics) > 5 else ''}")


Standard metrics (170) across 10 groups:
  Alternatives: breakeven_inflation, carry01, carry_accrued, collateral_coverage, collateral_value...
  Carry: breakeven, carry_total, coupon_income, funding_cost, implied_financing_rate...
  Credit: bucketed_cs01, bucketed_cs01_hazard, correlation01, cs01, cs01_hazard...
  Equity: constituent_count, constituent_delta, delta_vol, equity_dividend_yield, equity_forward_price...
  FX: base_amount, floating_first_accrual_factor, fx01, fx_delta, inverse_rate...
  Greeks: bucketed_vega, charm, color, cross_gamma_fx_rates, cross_gamma_fx_vol...
  Pricing: accrued, asw_market, asw_par, basis, clean_price...
  Rates: annuity, annuity_primary, annuity_reference, basis_par_spread, convexity_adjustment...
  Sensitivity: bucketed_dv01, collateral_haircut01, collateral_price01, conversion01, convexity_adjustment_risk...
  Structured Credit: cdr, clo_warf, cmbs_dscr, cpr, prepayment01...


## Shared market data

Every section below prices against **one** market snapshot built on **`as_of = 2025-01-15`**.

We start from `_shared.build_demo_market`, the canonical cross-asset market used throughout these notebooks: USD OIS discount, EUR OIS (for FX option foreign discounting), a USD SOFR 3M forward curve, a single-name `CORP-HAZARD` curve for the CDS, and the EUR/USD quote in the typed `FxMatrix`. Reusing it — rather than re-typing curve knots per notebook — is what keeps prices comparable across the curriculum.

On top of that we insert only what this tour additionally needs: the two flat `VolSurface` grids, the equity spot/dividend-yield scalars, and a WTI `PriceCurve` for the commodity section. All of it goes through the typed market API, so the serialized snapshot is ready for every pricer without JSON patching.

In [2]:
AS_OF_DATE = DEMO_AS_OF
AS_OF = AS_OF_DATE.isoformat()


def build_market_context_json(as_of: date) -> str:
    """Extend the shared demo market with the extras this tour needs.

    `build_demo_market` is the canonical cross-asset snapshot for the example
    notebooks: it already carries `USD-OIS`, `EUR-OIS`, `USD-SOFR-3M`,
    `CORP-HAZARD` and the EUR/USD quote. Starting from it means every notebook
    in the curriculum discounts off the *same* curves, so numbers are
    comparable across notebooks instead of drifting per file.

    Only what the shared market lacks is added here: two flat vol surfaces for
    the option sections, the equity spot/dividend pair the equity option
    resolves, and the WTI forward price curve used by the commodity section.
    """
    mc = build_demo_market(as_of)

    exp_eq = [0.25, 0.5, 1.0, 2.0]
    strikes_eq = [4000.0, 4300.0, 4500.0, 4700.0, 5000.0]
    exp_fx = [0.25, 0.5, 1.0, 2.0]
    strikes_fx = [0.9, 1.0, 1.1, 1.2, 1.3]
    mc.insert(
        VolSurface(
            "EQUITY-VOL",
            exp_eq,
            strikes_eq,
            [0.22] * (len(exp_eq) * len(strikes_eq)),
        )
    )
    mc.insert(
        VolSurface(
            "EURUSD-VOL",
            exp_fx,
            strikes_fx,
            [0.12] * (len(exp_fx) * len(strikes_fx)),
        )
    )
    mc.insert_price("EQUITY-SPOT", 4500.0, "USD")
    mc.insert_price("EQUITY-DIVYIELD", 0.015)
    mc.insert(
        PriceCurve(
            "WTI-FORWARD",
            as_of,
            [(0.0, 70.0), (1.0, 71.0)],
            day_count="act_365f",
        )
    )

    return mc.to_json()


MARKET_JSON = build_market_context_json(AS_OF_DATE)
ROWS: list[dict] = []

print("as_of:", AS_OF)
print("market JSON length (chars):", len(MARKET_JSON))
print("surfaces:", len(json.loads(MARKET_JSON)["surfaces"]))

as_of: 2025-01-15
market JSON length (chars): 35936
surfaces: 2


## 1. Deposit (rates / money market)

A **deposit** is the simplest discounting exercise: fixed coupon on a known day-count between `start_date` and `maturity`. Model: **`discounting`**.

We optionally run `validate_instrument_json` to pretty-print the canonical wire format.

In [3]:
deposit = json.dumps(
    {
        "type": "deposit",
        "spec": {
            "id": "DEP-1",
            "notional": {"amount": 1_000_000.0, "currency": "USD"},
            "start_date": "2025-01-15",
            "maturity": "2025-06-15",
            "day_count": "Act360",
            "quote_rate": 0.05,
            "discount_curve_id": "USD-OIS",
            "attributes": {},
        },
    }
)

canon = validate_instrument_json(deposit)
print("canonical instrument JSON (first 220 chars):\n", canon[:220], "…")

out = price_instrument_with_metrics(
    deposit, MARKET_JSON, AS_OF, model="discounting", metrics=["dv01"]
)
vr = ValuationResult.from_json(out)
print(vr)
print("dv01:", vr.get_metric("dv01"))

ROWS.append(
    {
        "class": "Rates",
        "instrument_id": vr.instrument_id,
        "price": vr.price,
        "ccy": vr.currency,
        "key_metric": "dv01",
        "metric_value": vr.get_metric("dv01"),
    }
)

canonical instrument JSON (first 220 chars):
 {"type":"deposit","spec":{"id":"DEP-1","notional":{"amount":"1000000","currency":"USD"},"start_date":"2025-01-15","maturity":"2025-06-15","day_count":"Act360","quote_rate":"0.05","discount_curve_id":"USD-OIS","attributes …
ValuationResult(id="DEP-1", price=1001984.4142, currency=USD, metrics=1)
dv01: -41.45195796870394


## 2. Interest rate swap (IRS)

Plain **fixed-for-floating** swap: fixed leg vs SOFR-linked float with `forward_curve_id`. Still **`discounting`** for the standard registry path.

**Seasoning:** if the floating leg has resets **before** `as_of`, the engine may require historical fixings. Here we use an **unseasoned** swap starting `2025-04-15`.

**Fixed rate:** The fixed coupon is set near **par (~4.5%)** for this OIS/SOFR snapshot so the swap PV is not dominated by a large off-market fixed leg (a **3%** coupon would be deeply off-market vs forwards and is fine for demos only).

In [4]:
irs = json.dumps(
    {
        "type": "interest_rate_swap",
        "spec": {
            "id": "IRS-5Y-USD",
            "notional": {"amount": 10_000_000.0, "currency": "USD"},
            "side": "pay",
            "fixed": {
                "discount_curve_id": "USD-OIS",
                "rate": 0.045,
                "frequency": {"count": 6, "unit": "months"},
                "day_count": "Thirty360",
                "bdc": "modified_following",
                "calendar_id": None,
                "stub": "None",
                "start": "2025-04-15",
                "end": "2030-04-15",
                "par_method": None,
                "compounding_simple": True,
            },
            "float": {
                "discount_curve_id": "USD-OIS",
                "forward_curve_id": "USD-SOFR-3M",
                "spread_bp": 0.0,
                "frequency": {"count": 3, "unit": "months"},
                "day_count": "Act360",
                "bdc": "modified_following",
                "calendar_id": None,
                "stub": "None",
                "reset_lag_days": 2,
                "start": "2025-04-15",
                "end": "2030-04-15",
                "compounding": "Simple",
            },
            "attributes": {},
        },
    }
)

out = price_instrument_with_metrics(
    irs, MARKET_JSON, AS_OF, model="discounting", metrics=["dv01"]
)
vr = ValuationResult.from_json(out)
print(vr)
print("dv01:", vr.get_metric("dv01"))
ROWS.append(
    {
        "class": "Rates",
        "instrument_id": vr.instrument_id,
        "price": vr.price,
        "ccy": vr.currency,
        "key_metric": "dv01",
        "metric_value": vr.get_metric("dv01"),
    }
)


ValuationResult(id="IRS-5Y-USD", price=183388.1454, currency=USD, metrics=1)
dv01: 4315.96027768136


## 3. Credit default swap (CDS)

Single-name **CDS**: premium leg vs protection leg tied to `credit_curve_id` (`CORP-HAZARD` here). Use model **`hazard_rate`**.

JSON **`convention`** uses snake_case (e.g. `isda_na`), not `IsdaNa`.

In [5]:
cds = json.dumps(
    {
        "type": "credit_default_swap",
        "spec": {
            "id": "CDS-CORP-5Y",
            "notional": {"amount": 10_000_000.0, "currency": "USD"},
            "side": "pay",
            "convention": "isda_na",
            "premium": {
                "start": "2025-03-20",
                "end": "2030-03-20",
                "frequency": {"count": 3, "unit": "months"},
                "stub": "ShortFront",
                "bdc": "following",
                "calendar_id": None,
                "day_count": "Act360",
                "spread_bp": 100.0,
                "discount_curve_id": "USD-OIS",
            },
            "protection": {
                "credit_curve_id": "CORP-HAZARD",
                "recovery_rate": 0.4,
                "settlement_delay": 3,
            },
            "pricing_overrides": {},
            "attributes": {},
        },
    }
)

out = price_instrument_with_metrics(
    cds, MARKET_JSON, AS_OF, model="hazard_rate", metrics=["cs01"]
)
vr = ValuationResult.from_json(out)
print(vr)
print("cs01:", vr.get_metric("cs01"))
ROWS.append(
    {
        "class": "Credit",
        "instrument_id": vr.instrument_id,
        "price": vr.price,
        "ccy": vr.currency,
        "key_metric": "cs01",
        "metric_value": vr.get_metric("cs01"),
    }
)


ValuationResult(id="CDS-CORP-5Y", price=203276.6681, currency=USD, metrics=2)
cs01: 4127.88175096245


## 4. Equity option

Vanilla **equity** call: `black76` model, `spot_id` / `vol_surface_id` / optional `div_yield_id` resolved from the market snapshot (`EQUITY-SPOT`, `EQUITY-VOL`, `EQUITY-DIVYIELD`).

The wire schema uses a **numeric** `strike` and `notional` as `Money` (not `contract_size`).

In [6]:
equity_option = json.dumps(
    {
        "type": "equity_option",
        "spec": {
            "id": "SPX-CALL-4500",
            "underlying_ticker": "SPX",
            "strike": 4500.0,
            "option_type": "call",
            "exercise_style": "european",
            "expiry": "2026-06-15",
            "notional": {"amount": 100.0, "currency": "USD"},
            "day_count": "Act365F",
            "settlement": "cash",
            "discount_curve_id": "USD-OIS",
            "spot_id": "EQUITY-SPOT",
            "vol_surface_id": "EQUITY-VOL",
            "div_yield_id": "EQUITY-DIVYIELD",
            "pricing_overrides": {},
            "attributes": {},
        },
    }
)

out = price_instrument_with_metrics(
    equity_option,
    MARKET_JSON,
    AS_OF,
    model="black76",
    metrics=["delta"],
)
vr = ValuationResult.from_json(out)
print(vr)
print("delta:", vr.get_metric("delta"))
ROWS.append(
    {
        "class": "Equity",
        "instrument_id": vr.instrument_id,
        "price": vr.price,
        "ccy": vr.currency,
        "key_metric": "delta",
        "metric_value": vr.get_metric("delta"),
    }
)


ValuationResult(id="SPX-CALL-4500", price=55144.7612, currency=USD, metrics=1)
delta: 60.57315603259317


## 5. FX option

EUR/USD **call**: domestic discount `USD-OIS`, foreign `EUR-OIS`, `vol_surface_id` `EURUSD-VOL`, and FX spot from the typed matrix.

Model: **`black76`**. We request **`delta`** here (registered for this pricer); it is expressed in the option’s quoting currency and scales with notional.

In [7]:
fx_option = json.dumps(
    {
        "type": "fx_option",
        "spec": {
            "id": "FXOPT-EURUSD-CALL",
            "base_currency": "EUR",
            "quote_currency": "USD",
            "strike": 1.12,
            "option_type": "call",
            "exercise_style": "european",
            "expiry": "2026-06-15",
            "day_count": "Act365F",
            "notional": {"amount": 1_000_000.0, "currency": "EUR"},
            "settlement": "cash",
            "domestic_discount_curve_id": "USD-OIS",
            "foreign_discount_curve_id": "EUR-OIS",
            "vol_surface_id": "EURUSD-VOL",
            "pricing_overrides": {},
            "attributes": {},
        },
    }
)

out = price_instrument_with_metrics(
    fx_option,
    MARKET_JSON,
    AS_OF,
    model="black76",
    metrics=["delta"],
)
vr = ValuationResult.from_json(out)
print(vr)
print("delta:", vr.get_metric("delta"))
ROWS.append(
    {
        "class": "FX",
        "instrument_id": vr.instrument_id,
        "price": vr.price,
        "ccy": vr.currency,
        "key_metric": "delta",
        "metric_value": vr.get_metric("delta"),
    }
)


ValuationResult(id="FXOPT-EURUSD-CALL", price=52367.3727, currency=USD, metrics=1)
delta: 469397.0486189149


## Mini-example: side-by-side results

The table below collects **instrument id**, **PV**, **currency**, and one **requested metric** per row. Metrics are not uniform across asset classes (rates `dv01`, credit `cs01`, options `delta`)—that is intentional: it mirrors how desks ask different risk questions.

In [8]:
hdr = f"{'class':<8} {'instrument_id':<16} {'price':>14} {'ccy':<4} {'metric':<10} {'value':>12}"
print(hdr)
print("-" * len(hdr))
for row in ROWS:
    p = row["price"]
    p_str = f"{p:,.4f}" if p is not None else "n/a"
    m = row["metric_value"]
    m_str = f"{m:,.6f}" if m is not None else "n/a"
    print(
        f"{row['class']:<8} {row['instrument_id']:<16} {p_str:>14} "
        f"{row['ccy']:<4} {row['key_metric']:<10} {m_str:>12}"
    )
print("rows:", len(ROWS))

class    instrument_id             price ccy  metric            value
---------------------------------------------------------------------
Rates    DEP-1            1,001,984.4142 USD  dv01         -41.451958
Rates    IRS-5Y-USD         183,388.1454 USD  dv01       4,315.960278
Credit   CDS-CORP-5Y        203,276.6681 USD  cs01       4,127.881751
Equity   SPX-CALL-4500       55,144.7612 USD  delta         60.573156
FX       FXOPT-EURUSD-CALL    52,367.3727 USD  delta      469,397.048619
rows: 5


## Commodity (forward)

Commodity instruments resolve a `PriceCurve` (or `ForwardCurve`) for the underlying plus a discount curve — both already inserted into the shared snapshot above, so this section only has to supply the instrument JSON.

In [9]:
_NOTEBOOK_DATA = json.loads(Path("data/pricing_across_asset_classes.json").read_text())

cf_json = json.dumps(_NOTEBOOK_DATA["commodity_fwd"])
print("validate:", validate_instrument_json(cf_json)[:120], "…")

# No new market needed: the shared snapshot already carries USD-OIS (discounting)
# and the WTI-FORWARD price curve this contract's `forward_curve_id` resolves.
out = price_instrument_with_metrics(cf_json, MARKET_JSON, AS_OF)
vr = ValuationResult.from_json(out)
print("JSON path PV:", round(vr.price, 2))
print("Canonical JSON path complete.")

validate: {"type":"commodity_forward","spec":{"id":"WTI-FWD-2025M03","commodity_type":"Energy","ticker":"CL","unit":"BBL","currenc …
JSON path PV: 0.0
Canonical JSON path complete.


## Takeaways

- **One pipeline:** `validate_instrument_json` → `price_instrument*` → `ValuationResult.from_json` works across rates, credit, equity, and FX; `02_pricing/pricing_fundamentals.ipynb` is the reference for that skeleton.
- **Model keys matter:** `discounting` for deposits and swaps, `hazard_rate` for CDS, `black76` for vanilla equity and FX options in the default registry wiring.
- **Market data is explicit:** curves must cover every `*_curve_id`; options need typed **surfaces** and **scalar prices**; FX options need the relevant pair in `FxMatrix`.
- **Deep dives:** follow the curriculum notebooks on **interest_rate_swap** / **credit_default_swap** / **equity_option** / **fx_option** instrument JSON (and the foundations notebook on **curves and `MarketContext`**) for fuller schedules, conventions, and metric sets.

**Cross-references (repo):** instrument schemas under `finstack-quant/valuations/schemas/instruments/`, worked JSON under `finstack-quant/valuations/tests/instruments/json_examples/`, and the market snapshot format in `finstack-quant/core` (`MarketContext` serde / `VolSurface` grid layout: `vols_row_major`).